# TransLink Forecasting Team: Pandas on Databricks
### Your pandas runs as-is. Moving to Databricks does not mean rewriting your code.

**Audience:** Forecasting & Modelling team (real estate value around SkyTrain stations), moving off Azure Batch + Docker.

**The one thing to leave with:** the code you already have keeps working. You add scale when you need it, with a one-line change, and only where it actually helps.

**Format:** hands-on, hackathon style, ~2 hours. Each person on their own notebook against this shared workspace. Run every cell, then swap in your own pandas where the notebook says so.

| Module | Topic | Time |
|---|---|---|
| 0 | Framing + the mental-model map | 15 min |
| 1 | Your first interactive notebook (the dev loop) | 40 min |
| 2 | Your data, where it lives | 30 min |
| 3 | Bringing your environment across (the container question) | 35 min |
| -- | Break | 10 min |
| 4 | Scaling without a rewrite (the important one) | 45 min |
| 5 | From notebook to scheduled job | 20 min |
| 6 | What's next + MCP | 15 min |

## Module 0: Framing and the mental-model map

Today you bake a Docker image with your packages, submit it to **Azure Batch**, it runs on one big VM, and you read the logs when it finishes. The container is your unit of work.

On Databricks the container stops being your unit of work. You work inside a notebook attached to compute, you see output as it runs, and you scale up (bigger machine) **or** out (more machines) without repackaging anything.

| What you do today (Azure Batch) | The Databricks equivalent |
|---|---|
| One big VM | A cluster (one node, or many) |
| Docker image with your packages | Cluster environment: `%pip`, `requirements.txt`, cluster libraries, serverless environments, or Databricks Container Services |
| Submit job and wait | Interactive notebook, run cell by cell |
| Scale = bigger VM | Scale up (bigger node) **or** scale out (more nodes) |
| Read logs after the run | Live output under each cell |
| `python forecast.py` entry point | Notebook, or a scheduled Workflow |

Everything below is just pandas plus a few Databricks conveniences. Nothing here asks you to learn Spark.

### Setup: generate a realistic dataset

We need data that looks like yours: daily ridership by SkyTrain station — the kind of Compass card tap data your Forecasting & Modelling team uses for demand planning, service optimization, and revenue projections.

The synthetic data covers all four SkyTrain lines (Expo, Millennium, Canada, Broadway Subway), 50 stations across 7 municipalities, daily granularity from 2018–2025, and reflects real ridership dynamics: COVID collapse, gradual recovery, weekday/weekend patterns, seasonal variation, and the Broadway Subway opening in 2025. Later you point the same code at your real data.

Set the catalog/schema once. Defaults are writable on this workshop workspace.

In [0]:
dbutils.widgets.text("catalog", "serverless_stable_l26d62_catalog", "Catalog")
dbutils.widgets.text("schema", "translink_pandas_workshop", "Schema")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
TABLE = f"{CATALOG}.{SCHEMA}.station_ridership"
print(f"Writing to {TABLE}")

In [0]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# --- Station metadata: line, municipality, zone, and typical daily boardings ---
# Base ridership calibrated to TransLink 2019 pre-COVID levels (avg weekday boardings)
STATION_META = {
    # Expo Line — Waterfront to King George
    "Waterfront":              {"line": "Expo",       "municipality": "Vancouver",       "zone": 1, "base_boardings": 28000},
    "Burrard":                 {"line": "Expo",       "municipality": "Vancouver",       "zone": 1, "base_boardings": 14000},
    "Granville":               {"line": "Expo",       "municipality": "Vancouver",       "zone": 1, "base_boardings": 12000},
    "Stadium-Chinatown":       {"line": "Expo",       "municipality": "Vancouver",       "zone": 1, "base_boardings":  9500},
    "Main St-Science World":   {"line": "Expo",       "municipality": "Vancouver",       "zone": 1, "base_boardings":  8000},
    "Commercial-Broadway":     {"line": "Expo",       "municipality": "Vancouver",       "zone": 1, "base_boardings": 24000},
    "Nanaimo":                 {"line": "Expo",       "municipality": "Vancouver",       "zone": 1, "base_boardings":  5500},
    "29th Avenue":             {"line": "Expo",       "municipality": "Vancouver",       "zone": 1, "base_boardings":  4800},
    "Joyce-Collingwood":       {"line": "Expo",       "municipality": "Vancouver",       "zone": 1, "base_boardings":  7200},
    "Patterson":               {"line": "Expo",       "municipality": "Burnaby",         "zone": 1, "base_boardings":  4200},
    "Metrotown":               {"line": "Expo",       "municipality": "Burnaby",         "zone": 1, "base_boardings": 16000},
    "Royal Oak":               {"line": "Expo",       "municipality": "Burnaby",         "zone": 1, "base_boardings":  4500},
    "Edmonds":                 {"line": "Expo",       "municipality": "Burnaby",         "zone": 1, "base_boardings":  5800},
    "22nd Street":             {"line": "Expo",       "municipality": "New Westminster", "zone": 1, "base_boardings":  4000},
    "New Westminster":         {"line": "Expo",       "municipality": "New Westminster", "zone": 1, "base_boardings":  7500},
    "Columbia":                {"line": "Expo",       "municipality": "New Westminster", "zone": 1, "base_boardings":  5200},
    "Scott Road":              {"line": "Expo",       "municipality": "Surrey",          "zone": 2, "base_boardings":  4800},
    "Gateway":                 {"line": "Expo",       "municipality": "Surrey",          "zone": 2, "base_boardings":  5500},
    "Surrey Central":          {"line": "Expo",       "municipality": "Surrey",          "zone": 2, "base_boardings":  8500},
    "King George":             {"line": "Expo",       "municipality": "Surrey",          "zone": 2, "base_boardings": 10000},
    # Millennium Line — VCC-Clark to Lafarge Lake-Douglas
    "VCC-Clark":               {"line": "Millennium", "municipality": "Vancouver",       "zone": 1, "base_boardings":  6500},
    "Renfrew":                 {"line": "Millennium", "municipality": "Vancouver",       "zone": 1, "base_boardings":  4200},
    "Rupert":                  {"line": "Millennium", "municipality": "Vancouver",       "zone": 1, "base_boardings":  3800},
    "Gilmore":                 {"line": "Millennium", "municipality": "Burnaby",         "zone": 1, "base_boardings":  3500},
    "Brentwood Town Centre":   {"line": "Millennium", "municipality": "Burnaby",         "zone": 1, "base_boardings":  8200},
    "Holdom":                  {"line": "Millennium", "municipality": "Burnaby",         "zone": 1, "base_boardings":  3000},
    "Sperling-Burnaby Lake":   {"line": "Millennium", "municipality": "Burnaby",         "zone": 1, "base_boardings":  2800},
    "Lake City Way":           {"line": "Millennium", "municipality": "Burnaby",         "zone": 1, "base_boardings":  3200},
    "Production Way-University":{"line": "Millennium", "municipality": "Burnaby",         "zone": 1, "base_boardings":  5800},
    "Lougheed Town Centre":    {"line": "Millennium", "municipality": "Burnaby",         "zone": 2, "base_boardings":  7500},
    "Burquitlam":              {"line": "Millennium", "municipality": "Coquitlam",       "zone": 2, "base_boardings":  4500},
    "Moody Centre":            {"line": "Millennium", "municipality": "Port Moody",      "zone": 2, "base_boardings":  3800},
    "Coquitlam Central":       {"line": "Millennium", "municipality": "Coquitlam",       "zone": 2, "base_boardings":  5200},
    "Lincoln":                 {"line": "Millennium", "municipality": "Coquitlam",       "zone": 2, "base_boardings":  3000},
    "Lafarge Lake-Douglas":    {"line": "Millennium", "municipality": "Coquitlam",       "zone": 2, "base_boardings":  4200},
    # Canada Line — Waterfront to Richmond-Brighouse / YVR
    "Yaletown-Roundhouse":     {"line": "Canada",    "municipality": "Vancouver",       "zone": 1, "base_boardings":  8500},
    "Olympic Village":         {"line": "Canada",    "municipality": "Vancouver",       "zone": 1, "base_boardings":  5200},
    "Broadway-City Hall":      {"line": "Canada",    "municipality": "Vancouver",       "zone": 1, "base_boardings": 11000},
    "King Edward":             {"line": "Canada",    "municipality": "Vancouver",       "zone": 1, "base_boardings":  4500},
    "Oakridge-41st Avenue":    {"line": "Canada",    "municipality": "Vancouver",       "zone": 1, "base_boardings":  5800},
    "Langara-49th Avenue":     {"line": "Canada",    "municipality": "Vancouver",       "zone": 1, "base_boardings":  6200},
    "Marine Drive":            {"line": "Canada",    "municipality": "Vancouver",       "zone": 1, "base_boardings":  4800},
    "Bridgeport":              {"line": "Canada",    "municipality": "Richmond",        "zone": 2, "base_boardings":  7800},
    "Aberdeen":                {"line": "Canada",    "municipality": "Richmond",        "zone": 2, "base_boardings":  5500},
    "Lansdowne":               {"line": "Canada",    "municipality": "Richmond",        "zone": 2, "base_boardings":  5200},
    "Richmond-Brighouse":      {"line": "Canada",    "municipality": "Richmond",        "zone": 2, "base_boardings":  8000},
    # Broadway Subway (Millennium extension, opened Jan 2025)
    "Great Northern Way":      {"line": "Broadway Subway", "municipality": "Vancouver", "zone": 1, "base_boardings":  6000},
    "Mount Pleasant":          {"line": "Broadway Subway", "municipality": "Vancouver", "zone": 1, "base_boardings":  7500},
    "South Granville":         {"line": "Broadway Subway", "municipality": "Vancouver", "zone": 1, "base_boardings":  8000},
    "Arbutus":                 {"line": "Broadway Subway", "municipality": "Vancouver", "zone": 1, "base_boardings":  9500},
}

# --- Ridership recovery curve (fraction of 2019 baseline) ---
# Reflects: pre-COVID normal, March 2020 collapse, gradual recovery, 2024-25 ~90-95% of 2019
def ridership_factor(date: pd.Timestamp) -> float:
    """Monthly ridership level relative to 2019 baseline."""
    if date < pd.Timestamp("2020-03-15"):
        return 1.0  # pre-COVID
    elif date < pd.Timestamp("2020-06-01"):
        return 0.20  # initial collapse (~80% drop)
    elif date < pd.Timestamp("2020-09-01"):
        return 0.35
    elif date < pd.Timestamp("2021-01-01"):
        return 0.40
    elif date < pd.Timestamp("2021-07-01"):
        return 0.50
    elif date < pd.Timestamp("2022-01-01"):
        return 0.60
    elif date < pd.Timestamp("2022-09-01"):
        return 0.70
    elif date < pd.Timestamp("2023-06-01"):
        return 0.78
    elif date < pd.Timestamp("2024-01-01"):
        return 0.85
    elif date < pd.Timestamp("2025-01-01"):
        return 0.90
    else:
        return 0.95  # 2025: near full recovery

# --- Generate daily ridership records ---
date_range = pd.date_range("2018-01-01", "2025-12-31", freq="D")

rows = []
for station, meta in STATION_META.items():
    base = meta["base_boardings"]
    # Broadway Subway only has data from 2025 onward
    if meta["line"] == "Broadway Subway":
        station_dates = date_range[date_range >= "2025-01-20"]
    else:
        station_dates = date_range

    for date in station_dates:
        # Day-of-week effect: weekday vs weekend
        dow = date.dayofweek
        if dow < 5:  # Mon-Fri
            dow_factor = 1.0
        elif dow == 5:  # Saturday
            dow_factor = 0.55
        else:  # Sunday
            dow_factor = 0.40

        # Seasonal pattern: slightly higher Sept-Nov (back to school/work), lower summer/holidays
        month = date.month
        seasonal = {1: 0.90, 2: 0.95, 3: 1.00, 4: 1.02, 5: 1.03, 6: 0.95,
                    7: 0.85, 8: 0.85, 9: 1.05, 10: 1.05, 11: 1.02, 12: 0.88}[month]

        # COVID / recovery
        covid = ridership_factor(date)

        # Compute boardings with noise
        boardings = int(
            base * dow_factor * seasonal * covid * rng.normal(1.0, 0.08)
        )
        boardings = max(0, boardings)

        # Fare product breakdown (proportional to boardings)
        adult_pct = rng.normal(0.55, 0.03)
        concession_pct = rng.normal(0.15, 0.02)  # youth + senior
        monthly_pass_pct = rng.normal(0.25, 0.03)
        daypass_pct = max(0, 1 - adult_pct - concession_pct - monthly_pass_pct)

        rows.append((
            station, meta["line"], meta["municipality"], meta["zone"],
            date, date.year, month, dow,
            "weekday" if dow < 5 else "weekend",
            boardings,
            int(boardings * adult_pct),
            int(boardings * concession_pct),
            int(boardings * monthly_pass_pct),
            int(boardings * daypass_pct),
        ))

data = pd.DataFrame(rows, columns=[
    "station", "skytrain_line", "municipality", "zone",
    "date", "year", "month", "day_of_week",
    "day_type",
    "total_boardings",
    "adult_boardings", "concession_boardings",
    "monthly_pass_boardings", "daypass_boardings",
])
data["date"] = pd.to_datetime(data["date"])

print(f"{len(data):,} daily records across {data['station'].nunique()} stations, "
      f"{data['skytrain_line'].nunique()} lines, {data['municipality'].nunique()} municipalities")
print(f"Date range: {data['date'].min().date()} to {data['date'].max().date()}")
print(f"\nTotal system boardings by year:")
print(data.groupby('year')['total_boardings'].sum().apply(lambda x: f"{x:,.0f}"))
print(f"\nSample:")
data.head(10)

In [0]:
# Persist it once as a Unity Catalog table so every module can read it back the same way your real data would live.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.createDataFrame(data).write.mode("overwrite").saveAsTable(TABLE)
print(f"Saved {TABLE}")

## Module 1: Your first interactive notebook (the dev loop)

This is the centerpiece of the morning. The point is muscle memory: edit a cell, run it, see output immediately. No image build, no submit, no waiting.

Two small differences from a script:
- `print(df)` gives you plain text. `display(df)` gives you a sortable, chartable grid. Use `display` for anything you want to look at.
- The last expression in a cell renders on its own, like in Jupyter.

In [0]:
# This is plain pandas. Nothing Databricks-specific. It runs on a single node, exactly like your laptop or one Azure Batch VM.
recent = data[data["year"] >= 2024]
by_line = recent.groupby(["skytrain_line", "day_type"])["total_boardings"].agg(["count", "mean", "median"]).round(0)
print(by_line)          # plain text

In [0]:
display(by_line.reset_index())   # rich, sortable, one click to a chart

**Your turn (5 min).** Paste a chunk of your own pandas below. A `read_csv` on a small file, a `groupby`, a feature you build every week. Confirm it runs here unchanged. If something complains, flag it; that is exactly the kind of gap this workshop exists to close.

In [0]:
# >>> Your pandas here <<<


## Module 2: Your data, where it lives

Three places your data will live on Databricks:


1. **Unity Catalog Volumes**: governed folders for files (CSV, Parquet, model artifacts, anything path-based). This is where your loose CSVs go.
2. **Unity Catalog Tables**: governed Delta tables you query by name. This is where curated data lands.

You read all three with code you already know.

In [0]:
# Pattern A: read files from a Volume with pandas, by path. This is your read_csv, just pointed at a governed folder.
# Example (pointed at your data later):
#   df = pd.read_csv("/Volumes/<catalog>/<schema>/<volume>/sales_2025.csv")
# It is ordinary pandas. The only change is the path.

# Pattern B: read a Unity Catalog table. Two equivalent ways:
sdf = spark.read.table(TABLE)         # a Spark DataFrame
df_from_table = sdf.toPandas()        # pull into pandas (fine when the result fits in memory)
print(type(df_from_table), df_from_table.shape)
df_from_table.head()

In [0]:
# Reading files from a Unity Catalog Volume — just like read_csv with a governed path.
# Volumes live at /Volumes/<catalog>/<schema>/<volume>/<file>

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_files"

# List what's in the volume
import os
print("Files in volume:")
for f in os.listdir(VOLUME_PATH):
    print(f"  {f}")

# Read a CSV — plain pandas, just a different path
df_csv = pd.read_csv(f"{VOLUME_PATH}/ridership_export_2025.csv")
print(f"\nCSV: {df_csv.shape[0]} rows, {df_csv.shape[1]} columns")
display(df_csv.head())

# Read a Parquet file (common for larger exports)
df_parquet = pd.read_parquet(f"{VOLUME_PATH}/station_history.parquet")
print(f"\nParquet: {df_parquet.shape[0]} rows, {df_parquet.shape[1]} columns")
display(df_parquet.head())

You can also run SQL and get a DataFrame back. Handy for a quick filter before you bring data into pandas, so you only pull what you need.

In [0]:
q = spark.sql(f"""
  SELECT station, skytrain_line, year, ROUND(AVG(total_boardings)) AS avg_daily_boardings, COUNT(*) AS days
  FROM {TABLE}
  WHERE year >= 2023
  GROUP BY station, skytrain_line, year
  ORDER BY avg_daily_boardings DESC
""")
display(q)

## Module 3: Bringing your environment across (the container question)

The real question behind "do we have to rewrite everything" is usually: **how do my packages come with me?** Today that is your Dockerfile. On Databricks there is a spectrum, lightest to heaviest:

1. **`%pip install`** in the notebook. Best for "I need one more library." Scoped to your session.
2. **`requirements.txt`** via `%pip install -r`. Your existing pin file works as-is.
3. **Serverless environments / cluster libraries**: declare packages once, attached for everyone.
4. **Init scripts**: shell setup at cluster start, for the odd system dependency.
5. **Databricks Container Services (DCS)**: bring your actual Docker image when you genuinely need it.

Start at the top. Most teams never need to go past `requirements.txt`.

**On DCS, so you can plan:** it is in Beta on Standard compute (DBR 18+). Your base image must extend `databricksruntime/environment:v5-standard`. Current limits: no compute-scoped libraries, no private package repos, and not supported on ML Runtime. So for most forecasting work, `%pip` plus a `requirements.txt` is the path, and DCS is the escape hatch for a hard system dependency.

In [0]:
# Your requirements.txt works unchanged. Uncomment to try one library.
# %pip install statsmodels==0.14.2
# After a %pip install, restart Python so imports pick it up:
# dbutils.library.restartPython()

## Module 4: Scaling without a rewrite (the important one)

Protect this section. This is where the "rewrite" fear actually gets resolved.

**Step 1. Single node first.** If your data fits in memory, you are done. Use pandas. Do not distribute anything. A single big node on Databricks is already bigger than most Azure Batch VMs.

**Step 2. When the data outgrows one node:** use the pandas API on Spark (`pyspark.pandas`). Change the import, keep the code. Same methods, same syntax — it just runs distributed across the cluster instead of on one machine.

### The pandas API on Spark: one-line scale-out

`import pyspark.pandas as ps` gives you the pandas API backed by Spark. Same methods, runs across the cluster, no out-of-memory wall.

In [0]:
import pyspark.pandas as ps

# Read the table straight into a pandas-on-Spark frame:
psdf = ps.read_table(TABLE)

# This is pandas syntax. It just runs distributed.
avg_by_station = psdf.groupby(["skytrain_line", "station"])["total_boardings"].mean().sort_values(ascending=False)
display(avg_by_station.head(20).reset_index())

In [0]:
# You can move back and forth. A Spark DataFrame -> pandas-on-Spark in one call:
psdf2 = spark.read.table(TABLE).pandas_api()
print(type(psdf2))
# And a small result back to real pandas when you want plain pandas in hand:
small = psdf.groupby("skytrain_line")["total_boardings"].mean().to_pandas()
print(small)

**Three gotchas with pandas-on-Spark, so they do not surprise you:**
- It is not 100% identical to pandas. Most things match; a few methods differ or are missing. Check the error, not your sanity.
- It is lazy. Work is deferred until you display or call `.to_pandas()`. That is a feature; it lets Spark optimize.
- There is a default index. If row order or the index matters to you, set it explicitly.

That is the whole scaling story. Same pandas syntax, distributed when you need it.

**The decision is simple:**
- Data fits in memory on one node → plain pandas. Do not over-engineer.
- Data too big for one node, or you want the cluster to handle it → `pyspark.pandas`.
- Each model is heavy and independent and you want retries/observability per model → a Workflow with a for-each task (Module 5).

### A quick chart, the pandas way
pandas-on-Spark plotting uses interactive Plotly. Standard `.plot`, no new API.

In [0]:
import pyspark.pandas as ps
chart = ps.read_table(TABLE)
pivot = chart.groupby(["year", "skytrain_line"])["total_boardings"].mean().unstack()
pivot.plot.line(title="Average daily boardings by SkyTrain line (2018–2025)")

## Module 5: From notebook to scheduled job

When the notebook is right, schedule it. No rewrite to a different artifact.

- **Workflows (Lakeflow Jobs):** point a job at this notebook, give it a schedule, pick the compute. A **for-each task** can run independent work items as parallel tasks if you want per-item retries and logs.
- **Job compute vs interactive:** develop on interactive compute (this notebook), run scheduled work on job compute. It spins up for the run and shuts down after, so you are not paying for an idle interactive cluster.
- **For Brandon's team / version control:** job definitions live in code via **Databricks Asset Bundles** (a YAML definition of the job + notebook, checked into git). That is the answer to "how do we version-control jobs coming from Synapse." Separate POV coming, but the short version: the bundle is your source of truth, deployed through CI.

## Module 6: What's next

**You proved today:** your pandas runs unchanged; data reads with code you know; environments come across without Docker in most cases; and scale is a one-line import change (`pyspark.pandas`) only where it actually helps.

**Suggested next steps for the team:**
1. Each person: get one real forecasting script running in a notebook against a real input. That is the whole goal.
2. Pick one ridership aggregation or demand model that runs slow on a single node and try it with `pyspark.pandas`. Measure wall-clock before and after.
3. Move one curated dataset into a UC table and one folder of CSVs into a Volume.
4. Brandon's team: align on Asset Bundles for job version control (Synapse to Databricks).

### Optional cleanup
Drops the workshop table. Leave commented if attendees still want to explore.

In [0]:
# spark.sql(f"DROP TABLE IF EXISTS {TABLE}")
# print(f"Dropped {TABLE}")